# 03 — Style de jeu des équipes : démarche critique

**Question.** Peut-on définir le *style* d'une équipe, et un style en bat-il un autre ?

Ce notebook montre une **erreur méthodologique fréquente puis sa correction** — c'est
le cœur de la rigueur du projet.

- **Approche naïve (rejetée).** Clusteriser sur eFG%, rebond, etc. → on ne sépare que
  *les forts des faibles* (ces variables mesurent la **qualité**, pas le style).
- **Approche corrigée.** Distinguer **niveau** (à quel point on est fort) et **style**
  (*comment* on joue), avec des variables neutres au niveau, puis **tester** si le style
  sert à quelque chose une fois le niveau (ELO) connu.

In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "analyse" / "outils.py").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p / "analyse"))
import numpy as np, pandas as pd, outils as o
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
OUT = o.RESULTATS / "styles"; OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 30)
print("sorties ->", OUT)

sorties -> C:\Users\TEMP.IUT-LUMIERE.000\R6.06-Domaines-d-application-de-la-statistique\analyse\resultats\styles


## Données de style : d'où viennent les points

On exploite une donnée jamais utilisée ailleurs : la ligne **`TOTAUX EQUIPE`** du fichier
stats, qui donne la **répartition des points** (raquette, contre-attaque, 2e chance, banc)
— c'est du *comment*, pas du *combien*. + rythme, part de 3pts, taille des joueuses,
défense agressive, dépendance star.

In [2]:
df = o.charger_equipes(o.SAISONS_COMPLETES)
tot = df[df["Joueur"] == "TOTAUX EQUIPE"].copy()
rep = tot.groupby(["eq","Saison"]).agg(PTS=("Pts","sum"), Pint=("Points_int","sum"),
        P2c=("Point_2eme_chance","sum"), PCA=("Points_CA","sum"),
        Pbanc=("Points_banc","sum")).reset_index()

j = df[df["Joueur"] != "TOTAUX EQUIPE"]
keys = ["eq","Saison","Num_match","dom_ext","adv"]
tm = j.groupby(keys, as_index=False).agg(FGA=("Tirs_tentes","sum"), TPA=("3pts_tentes","sum"),
        TPM=("3pts_marques","sum"), FTA=("LF_tentes","sum"), RO=("RO","sum"),
        BP=("BP","sum"), INT=("INT","sum"), Pts=("Pts","sum"))
star = (j.groupby(keys)["Pts"].max()/j.groupby(keys)["Pts"].sum()).rename("dep").reset_index()
tm = tm.merge(star, on=keys)
res = j.groupby(keys)["Gagne_perdu"].first().reset_index()
res["v"] = res["Gagne_perdu"].str.strip().str.lower().eq("victoire").astype(int)
tm = tm.merge(res, on=keys); tm["poss"] = tm["FGA"]-tm["RO"]+tm["BP"]+0.44*tm["FTA"]
agg = tm.groupby(["eq","Saison"]).agg(matchs=("v","size"), wr=("v","mean"),
        FGA=("FGA","sum"), TPA=("TPA","sum"), TPM=("TPM","sum"), poss=("poss","sum"),
        INT=("INT","sum"), dep_star=("dep","mean"), pace=("poss","mean")).reset_index()

jt = j.copy(); jt["Tps_jeu_decimal"]=pd.to_numeric(jt["Tps_jeu_decimal"],errors="coerce")
jt = jt.merge(o.tailles_par_cle(), on=["cle","Saison"], how="left")
jt["mt"] = jt["Tps_jeu_decimal"]*jt["taille"]
th = (jt.groupby(["eq","Saison"]).apply(lambda x: x["mt"].sum()/
        x.loc[x["taille"].notna(),"Tps_jeu_decimal"].sum()).rename("taille_moy").reset_index())

a = agg.merge(rep, on=["eq","Saison"]).merge(th, on=["eq","Saison"])
eps = 1e-9
a["pct_raquette"]   = a["Pint"]/(a["PTS"]+eps)
a["pct_3pts_pts"]   = 3*a["TPM"]/(a["PTS"]+eps)
a["pct_CA"]         = a["PCA"]/(a["PTS"]+eps)
a["pct_2e_chance"]  = a["P2c"]/(a["PTS"]+eps)
a["pct_banc"]       = a["Pbanc"]/(a["PTS"]+eps)
a["part_3pts_tirs"] = a["TPA"]/(a["FGA"]+eps)
a["gambling_def"]   = a["INT"]/(a["poss"]+eps)
print(f"{len(a)} equipe-saisons profilees\n")
print(a[["eq","Saison","wr","pct_raquette","pct_3pts_pts","pct_CA","pace","taille_moy"]].round(2).head().to_string(index=False))

20 equipe-saisons profilees

           eq Saison   wr  pct_raquette  pct_3pts_pts  pct_CA  pace  taille_moy
       angers  23-24 0.41          0.38          0.27    0.12 75.86        1.82
       angers  24-25 0.55          0.46          0.21    0.13 74.69        1.82
basket landes  23-24 0.73          0.46          0.24    0.15 76.93        1.83
basket landes  24-25 0.68          0.45          0.30    0.15 76.10        1.83
      bourges  23-24 0.82          0.45          0.25    0.17 73.56        1.82


## Étape clé : le style est-il indépendant du niveau ?

Corrélation de chaque variable candidate avec le taux de victoire. |r| faible = vraie
variable de **style** ; |r| fort = elle mesure en réalité le **niveau** → on l'écarte.

In [3]:
cands = ["pct_raquette","pct_3pts_pts","pct_CA","pct_2e_chance","pct_banc",
         "part_3pts_tirs","pace","taille_moy","dep_star","gambling_def"]
corr = a[cands].corrwith(a["wr"]).sort_values(key=abs).round(2)
print("corr(variable, taux de victoire) :")
print(corr.to_string())
style_pur = [c for c in cands if abs(a[c].corr(a["wr"])) < 0.45]
print("\n-> variables de STYLE retenues (|r|<0.45) :", style_pur)

corr(variable, taux de victoire) :
pct_CA            0.03
part_3pts_tirs    0.05
taille_moy        0.11
pct_3pts_pts      0.15
pace             -0.24
pct_raquette      0.25
pct_banc          0.27
pct_2e_chance     0.29
gambling_def      0.34
dep_star         -0.69

-> variables de STYLE retenues (|r|<0.45) : ['pct_raquette', 'pct_3pts_pts', 'pct_CA', 'pct_2e_chance', 'pct_banc', 'part_3pts_tirs', 'pace', 'taille_moy', 'gambling_def']


**Découverte.** La **dépendance à la star** n'est pas un style mais un marqueur de
faiblesse (r très négatif) → écartée. C'est exactement le piège de l'approche naïve.

## Combien de styles ? (validation par silhouette)

In [4]:
X = StandardScaler().fit_transform(a[style_pur].values)
for k in range(2,6):
    km = KMeans(n_clusters=k, n_init=25, random_state=0).fit(X)
    print(f"k={k} : silhouette = {silhouette_score(X, km.labels_):.3f}")

k=2 : silhouette = 0.159
k=3 : silhouette = 0.162
k=4 : silhouette = 0.152
k=5 : silhouette = 0.187


**Lecture.** Toutes les silhouettes sont faibles (~0.16-0.18) : **il n'existe pas de
familles de style nettes** en LFB, les équipes forment un continuum. (L'approche naïve
imposait au contraire 4 archétypes francs — une sur-interprétation.) On garde le meilleur k.

In [5]:
best_k = max(range(2,6), key=lambda k: silhouette_score(X, KMeans(n_clusters=k,n_init=25,random_state=0).fit_predict(X)))
a["style"] = KMeans(n_clusters=best_k, n_init=30, random_state=0).fit_predict(X)
a.to_csv(OUT / "profils_style.csv", index=False)
for c in sorted(a["style"].unique()):
    sub = a[a["style"]==c]
    print(f"Style {c}: n={len(sub)}, wr de {sub['wr'].min()*100:.0f}% a {sub['wr'].max()*100:.0f}% (moy {sub['wr'].mean()*100:.0f}%)")
print("\n-> wr tres variable DANS chaque style => on a bien isole le COMMENT, pas le niveau.")

Style 0: n=5, wr de 14% a 82% (moy 55%)
Style 1: n=1, wr de 33% a 33% (moy 33%)
Style 2: n=4, wr de 27% a 86% (moy 53%)
Style 3: n=5, wr de 19% a 86% (moy 45%)
Style 4: n=5, wr de 41% a 68% (moy 55%)

-> wr tres variable DANS chaque style => on a bien isole le COMMENT, pas le niveau.


## Test décisif : le style sert-il à prédire qui gagne ?

On prédit chaque match avec **(1)** l'écart d'ELO seul, puis **(2)** ELO + différentiel
de style. Si l'AUC ne progresse pas, le style n'apporte rien une fois le niveau connu.

In [6]:
cal = o.charger_calendrier(forme=False)
st = a.set_index(["eq","Saison"])[style_pur]
rows = []
for _, r in cal.iterrows():
    kd, ke = (r["dom"], r["Saison"]), (r["ext"], r["Saison"])
    if kd in st.index and ke in st.index:
        d = (st.loc[kd] - st.loc[ke]).to_dict(); d.update(d_elo=r["d_elo"], home_win=r["home_win"])
        rows.append(d)
D = pd.DataFrame(rows).dropna(); y = D["home_win"].values
auc1 = cross_val_score(LogisticRegression(max_iter=2000),
        StandardScaler().fit_transform(D[["d_elo"]]), y, cv=5, scoring="roc_auc").mean()
auc2 = cross_val_score(LogisticRegression(max_iter=2000),
        StandardScaler().fit_transform(D[["d_elo"]+style_pur]), y, cv=5, scoring="roc_auc").mean()
print(f"{len(D)} matchs")
print(f"  Modele 1 (ELO seul)     AUC = {auc1:.3f}")
print(f"  Modele 2 (ELO + style)  AUC = {auc2:.3f}")
print(f"  gain du style           {auc2-auc1:+.3f}")

90 matchs
  Modele 1 (ELO seul)     AUC = 0.802
  Modele 2 (ELO + style)  AUC = 0.773
  gain du style           -0.029


## Conclusion

**Le style n'ajoute rien (voire dégrade) une fois l'ELO connu.** Le résultat d'un match
se joue sur le **niveau**, pas sur le style. L'intuition « on n'a finalement que : les
forts battent les faibles » est ainsi **démontrée** proprement.

Cela ne rend pas le style inutile : il sert à **décrire** une équipe et à **réfléchir au
recrutement**, mais **pas** à pronostiquer. Le style des *joueuses*, lui, est pertinent
→ notebook 04. Résultats dans `resultats/styles/`.